# Task 1 — Pure Synchronous Execution

In [1]:
import time
import asyncio
import threading

In [9]:
def run_sync_tasks():
    start=time.time()
    for i in range(1,9):
        print(f"task {i} started")
        time.sleep(1)
        print(f"task {i} completed")

    end=time.time()-start
    print(f"Total time taken to complete all tasks is {end} seconds")
    
   




In [10]:
run_sync_tasks()

task 1 started
task 1 completed
task 2 started
task 2 completed
task 3 started
task 3 completed
task 4 started
task 4 completed
task 5 started
task 5 completed
task 6 started
task 6 completed
task 7 started
task 7 completed
task 8 started
task 8 completed
Total time taken to complete all tasks is 8.008306503295898 seconds


# Task 2 — Async Sequential (Without gather)

In [16]:
 async def run_async_tasks():
    start=time.time()
    for i in range(1,9):
        print(f"task {i} started")
        await asyncio.sleep(1)
        print(f"task {i} completed")

    end=time.time()-start
    print(f"Total time taken to complete all tasks is {end} seconds")

In [18]:
 await run_async_tasks()

task 1 started
task 1 completed
task 2 started
task 2 completed
task 3 started
task 3 completed
task 4 started
task 4 completed
task 5 started
task 5 completed
task 6 started
task 6 completed
task 7 started
task 7 completed
task 8 started
task 8 completed
Total time taken to complete all tasks is 8.062145471572876 seconds


# Task 3 — Async Concurrent Using gather()

In [32]:
 async def run_async_tasks(name):
        print(f"task {name} started")
        await asyncio.sleep(1)
        print(f"task {name} completed")


In [35]:
async def main():
    start=time.time()
    await asyncio.gather(run_async_tasks('Task 1'), run_async_tasks('Task 2'), run_async_tasks('Task 3'), run_async_tasks('Task 4'), run_async_tasks('Task 5'), run_async_tasks('Task 6'), run_async_tasks('Task 7'), run_async_tasks('Task 8'))
    end=time.time()-start
    print(f"Total time taken to complete all tasks is {end} seconds")


In [36]:
await main()

task Task 1 started
task Task 2 started
task Task 3 started
task Task 4 started
task Task 5 started
task Task 6 started
task Task 7 started
task Task 8 started
task Task 1 completed
task Task 3 completed
task Task 7 completed
task Task 6 completed
task Task 8 completed
task Task 5 completed
task Task 2 completed
task Task 4 completed
Total time taken to complete all tasks is 1.000420331954956 seconds


# Task 4 — Blocking CPU Inside Event Loop

In [37]:
def heavy_cpu_task():
    print("Starting heavy CPU task...")
    total = 0
    for i in range(10**7):
        total += i
    print(f"Heavy CPU task completed.")

In [52]:
 async def run_async_tasks(name):
        print(f"task {name} started")
        await asyncio.sleep(3)
        print(f"task {name} completed")

In [ ]:
async def main():
    heavy_cpu_task()
    
    await asyncio.gather(run_async_tasks('Task 1'), run_async_tasks('Task 2'), run_async_tasks('Task 3'))

In [48]:
async def main2():
    task1=asyncio.create_task(run_async_tasks('Task 1'))
    task2= asyncio.create_task(run_async_tasks('Task 2'))

    await task1
    heavy_cpu_task()
    await task2

In [51]:
await main()

Starting heavy CPU task...
Heavy CPU task completed.
task Task 1 started
task Task 2 started
task Task 3 started
task Task 1 completed
task Task 2 completed
task Task 3 completed


In [49]:
await main2()

task Task 1 started
task Task 2 started
task Task 1 completed
task Task 2 completed
Starting heavy CPU task...
Heavy CPU task completed.


# Task 5 — Yielding Inside Single Thread

In [86]:
 async def run_async_tasks(name):
        print(f"task {name} started")
        await asyncio.sleep(0)
        print(f"task {name} completed")
        
        

In [87]:
async def main():
    stime=time.time()
    await asyncio.gather(run_async_tasks('Task 1'), run_async_tasks('Task 2'), run_async_tasks('Task 3'))
    print (time.time()-stime)


In [88]:
await main()

task Task 1 started
task Task 2 started
task Task 3 started
task Task 1 completed
task Task 2 completed
task Task 3 completed
0.0


In [84]:
async def task(name):
    for i in range(3):
        print(name, i)
        await asyncio.sleep(0)


async def main():
    stime=time.time()
    heavy_cpu_task()
    await asyncio.gather(
        task("Task 1"),
        task("Task 2")
    )
    
    print (time.time()-stime)

In [85]:
await main()

Starting heavy CPU task...
Heavy CPU task completed.
Task 1 0
Task 2 0
Task 1 1
Task 2 1
Task 1 2
Task 2 2
0.46921777725219727


# Task 6 — Using asyncio.to_thread()

In [96]:
# it allows us to run the heavy cpu task in a separate thread while the async tasks are running concurrently.
# like having a separate workerthread for the heavy cpu task and the async tasks running concurrently in the main thread.
async def main():

    task1=asyncio.create_task(run_async_tasks('Task 1'))
    task2= asyncio.create_task(run_async_tasks('Task 2'))
    cpu_task=asyncio.to_thread(heavy_cpu_task)
 
    await cpu_task
    await task1
    await task2

In [97]:
await main()

Starting heavy CPU task...task Task 1 started
task Task 2 started

task Task 1 completed
task Task 2 completed
Heavy CPU task completed.


# Task 7 — Race Condition Simulation

In [9]:
shared_counter=0

def increment():
    global shared_counter
    for i in range(10000000):
        shared_counter += 1

In [10]:
# without locks
# without locks the 2 threads will access same variable and every one will try to increment the value of shared_counter at the same time. This will lead to a race condition.
# and we want to make 2 threads complete each increment first.
t1=threading.Thread(target=increment)
t2=threading.Thread(target=increment)
t1.start()
t2.start()
#join() wait for the thread to complete before moving on to the next line of code.
#but this not mean that the thread 1 will finish first and then thread 2 will start. It just means that the main thread will wait for both threads to complete before moving on to the next line of code.
#and maybe th2 be also started with th1
t1.join()
t2.join()

print(f"shared_counter without locks: {shared_counter}")


shared_counter without locks: 20000000


In [ ]:
# with locks
# locks ensure that only one thread can access the shared variable at a time, and thread 2 will wait for thread 1 to finish its increment before it can access the shared variable.
shared_counter=0
lock=threading.Lock()

def increment_with_lock():
    global shared_counter
    for i in range(10000000):
        with lock:
            shared_counter += 1   

In [5]:
t1=threading.Thread(target=increment_with_lock)
t2=threading.Thread(target=increment_with_lock)
t1.start()
t2.start()
t1.join()
t2.join()

print(f"shared_counter with locks: {shared_counter}")

shared_counter with locks: 20000000


# Task 8 — Async + Thread Synchronization Hybrid

In [15]:
shared_counter=0
lock=threading.Lock()
# cpu work inside a thread
def cpu_work():
    global shared_counter
    for i in range(10000000):
        with lock:
            shared_counter += 1 

In [18]:
# async tasks
async def run_async_tasks(name):
    print(f"task {name} started")
    await asyncio.to_thread(cpu_work)
    print(f"task {name} completed")

In [19]:
async def main():
    await asyncio.gather(run_async_tasks('Task 1'), run_async_tasks('Task 2'), run_async_tasks('Task 3'))
    print(f"shared_counter after async tasks: {shared_counter}")

In [20]:
await main()

task Task 1 started
task Task 2 started
task Task 3 started
task Task 1 completed
task Task 2 completed
task Task 3 completed
shared_counter after async tasks: 30000000
